# 🔬 Majorana Qubits — Topological Quantum Computing

> *"The question is whether, in the language of theoretical physics, there exist
> neutral particles that are their own antiparticles."*
> — Ettore Majorana (1937)

This notebook provides a rigorous exploration of **Majorana fermions** and their
application to **topological quantum computing** — a fundamentally different approach
to building fault-tolerant qubits.

**What you will gain:**
- Mathematical construction of Majorana operators and their Clifford algebra
- The Dirac–Majorana correspondence and Jordan-Wigner transformation
- Kitaev's 1D chain model: exact solution, topological phases, and zero modes
- How Majorana zero modes encode topologically protected qubits
- Non-Abelian braiding statistics and their connection to quantum gates
- Topological protection against local perturbations
- The winding number as a topological invariant

> **Related notebooks:** For quantum gate fundamentals see the 100-series.
> For Hamiltonians and Ising models see `200_quantum_hamiltonian.ipynb`.

---
### Setup & Imports

*Prerequisites: linear algebra and quantum mechanics basics (100-series).
Notebook 200 provides background on Hamiltonians and spectral analysis.*

This notebook uses **NumPy** for operator algebra, **SciPy** for linear algebra, and
**Matplotlib** for visualization. Unlike the gate-based notebooks, we work directly
with fermionic Hamiltonians — no Qiskit circuits needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import reduce

# ── Pauli matrices ──────────────────────────────────────────────
I2 = np.eye(2, dtype=complex)
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

def kron_list(matrices):
    """Tensor product of a list of matrices."""
    return reduce(np.kron, matrices)

print("✅ Setup complete — Pauli basis ready")

---
## §1 — Majorana Fermions

In 1937, Ettore Majorana proposed a real solution to the Dirac equation: particles that
are their own antiparticles. In condensed matter physics, **Majorana fermions** emerge as
quasiparticle excitations in topological superconductors.

### Defining Properties

A set of **Majorana operators** $\{\gamma_0, \gamma_1, \ldots, \gamma_{2N-1}\}$
satisfies the **Clifford algebra**:

$$
\gamma_i^\dagger = \gamma_i \qquad \text{(self-adjoint: particle = antiparticle)}
$$

$$
\{\gamma_i, \gamma_j\} \equiv \gamma_i \gamma_j + \gamma_j \gamma_i = 2\delta_{ij} I
$$

Key consequences:
- $\gamma_i^2 = I$ (involutory)
- $\gamma_i \gamma_j = -\gamma_j \gamma_i$ for $i \neq j$ (anticommuting)
- The operators generate a $2^N$-dimensional Clifford algebra for $2N$ modes

### Jordan-Wigner Construction

For $N$ fermionic sites, we construct $2N$ Majorana operators using the
**Jordan-Wigner transformation**:

$$
\gamma_{2j} = \left(\bigotimes_{k<j} \sigma_z^{(k)}\right) \otimes \sigma_x^{(j)} \otimes \left(\bigotimes_{k>j} I^{(k)}\right)
$$

$$
\gamma_{2j+1} = \left(\bigotimes_{k<j} \sigma_z^{(k)}\right) \otimes \sigma_y^{(j)} \otimes \left(\bigotimes_{k>j} I^{(k)}\right)
$$

The $\sigma_z$ string ensures the correct fermionic anticommutation across sites.

In [ ]:
def build_majorana_ops(n_sites):
    """Construct 2*n_sites Majorana operators via Jordan-Wigner transform.

    gamma_{2j}   = Z x ... x Z x X x I x ... x I   (j copies of Z before X)
    gamma_{2j+1} = Z x ... x Z x Y x I x ... x I   (j copies of Z before Y)
    """
    ops = []
    for j in range(n_sites):
        chain_x = [sigma_z]*j + [sigma_x] + [I2]*(n_sites - j - 1)
        chain_y = [sigma_z]*j + [sigma_y] + [I2]*(n_sites - j - 1)
        ops.append(kron_list(chain_x))
        ops.append(kron_list(chain_y))
    return ops

# Build Majorana operators for N=2 sites (4 Majorana modes, dim=4)
n_sites = 2
gammas = build_majorana_ops(n_sites)
n_gamma = len(gammas)
dim = 2**n_sites

# ── Verify self-adjointness ──
for i in range(n_gamma):
    assert np.allclose(gammas[i], gammas[i].conj().T), f"gamma_{i} not Hermitian"

# ── Verify Clifford algebra: {gamma_i, gamma_j} = 2 delta_{ij} I ──
for i in range(n_gamma):
    for j in range(n_gamma):
        anticomm = gammas[i] @ gammas[j] + gammas[j] @ gammas[i]
        expected = 2 * np.eye(dim) if i == j else np.zeros((dim, dim))
        assert np.allclose(anticomm, expected), f"anticomm gamma_{i}, gamma_{j} failed"

print(f"Built {n_gamma} Majorana operators for {n_sites} sites (dim = {dim})")
print(f"✅ Self-adjointness verified: gamma_i† = gamma_i for all i")
print(f"✅ Clifford algebra verified: {{gamma_i, gamma_j}} = 2 delta_ij I")

---
## §2 — Dirac–Majorana Correspondence

Every ordinary (Dirac) fermion can be decomposed into two Majorana modes, and vice versa.
Given Majorana operators $\gamma_{2j}$ and $\gamma_{2j+1}$, define:

$$
c_j = \frac{\gamma_{2j} + i\gamma_{2j+1}}{2}, \qquad
c_j^\dagger = \frac{\gamma_{2j} - i\gamma_{2j+1}}{2}
$$

These satisfy the canonical anticommutation relations:

$$
\{c_i, c_j^\dagger\} = \delta_{ij}, \qquad \{c_i, c_j\} = 0
$$

The **number operator** $\hat{n}_j = c_j^\dagger c_j$ has eigenvalues $\{0, 1\}$,
representing the absence or presence of a fermion at site $j$.

Conversely, the Majorana operators are the "real" and "imaginary" parts of the Dirac
fermion:

$$
\gamma_{2j} = c_j + c_j^\dagger, \qquad \gamma_{2j+1} = -i(c_j - c_j^\dagger)
$$

This decomposition is the key insight: a single complex fermion degree of freedom splits
into two real (Majorana) degrees of freedom. In a topological superconductor, these two
halves can be **spatially separated** — this is what makes Majorana qubits special.

In [ ]:
# ── Reconstruct Dirac fermion from Majorana pair ──
c0     = (gammas[0] + 1j * gammas[1]) / 2
c0_dag = (gammas[0] - 1j * gammas[1]) / 2

# Verify canonical anticommutation relations
assert np.allclose(c0 @ c0_dag + c0_dag @ c0, np.eye(dim)), "{c, c_dag} != I"
assert np.allclose(c0 @ c0 + c0 @ c0, np.zeros((dim, dim))), "{c, c} != 0"

# Number operator n = c†c
n_op = c0_dag @ c0
eig_n = np.sort(np.linalg.eigvalsh(n_op).real)
unique_eig = sorted(set(np.round(eig_n, 10)))
print(f"Number operator eigenvalues: {unique_eig}")
assert set(np.round(eig_n, 5)) == {0.0, 1.0}, "n should have eigenvalues {0, 1}"

# Verify the inverse mapping
assert np.allclose(c0 + c0_dag, gammas[0]), "c + c_dag != gamma_0"
assert np.allclose(-1j * (c0 - c0_dag), gammas[1]), "-i(c - c_dag) != gamma_1"

print("✅ Dirac–Majorana correspondence verified:")
print("   {c, c†} = I,  {c, c} = 0,  n_hat eigenvalues in {0, 1}")
print("   c + c† = gamma_0,  -i(c - c†) = gamma_1")

---
## §3 — The Kitaev Chain

Alexei Kitaev's **1D p-wave superconductor** (2001) is the simplest model exhibiting
topological superconductivity with Majorana zero modes.

### Hamiltonian

$$
H = \sum_{j=0}^{N-2} \left[ -t\,(c_j^\dagger c_{j+1} + \text{h.c.})
+ \Delta\,(c_j c_{j+1} + \text{h.c.}) \right]
- \mu \sum_{j=0}^{N-1} c_j^\dagger c_j
$$

where:
- $\mu$ = chemical potential
- $t$ = nearest-neighbour hopping amplitude
- $\Delta$ = p-wave superconducting pairing potential

### Bogoliubov–de Gennes (BdG) Formalism

Using the Nambu spinor $\Psi = (c_0, \ldots, c_{N-1}, c_0^\dagger, \ldots, c_{N-1}^\dagger)^T$:

$$
H = \tfrac{1}{2}\, \Psi^\dagger \, H_{\text{BdG}} \, \Psi + \text{const.}
$$

$$
H_{\text{BdG}} = \begin{pmatrix} h & \Delta_{\text{mat}} \\ -\Delta_{\text{mat}}^* & -h^T \end{pmatrix}
$$

where $h$ is the single-particle Hamiltonian and $\Delta_{\text{mat}}$ is the pairing matrix.
The BdG spectrum has particle-hole symmetry: eigenvalues come in $\pm E$ pairs.
**Zero-energy eigenvalues signal Majorana zero modes.**

### Two Limiting Cases

1. **Trivial phase** ($\mu \to -\infty$, $t = \Delta = 0$): Majorana modes pair
   within each site — no edge modes.
2. **Topological phase** ($\mu = 0$, $t = \Delta$): Majorana modes pair between
   adjacent sites, leaving **unpaired edge modes** $\gamma_0$ and $\gamma_{2N-1}$.

In [ ]:
def kitaev_bdg(n_sites, mu, t, delta):
    """Build the BdG Hamiltonian for the Kitaev chain.

    Returns a 2N x 2N matrix with particle-hole symmetry.
    """
    N = n_sites
    h = np.zeros((N, N), dtype=complex)
    Delta_mat = np.zeros((N, N), dtype=complex)

    for j in range(N):
        h[j, j] = -mu
    for j in range(N - 1):
        h[j, j+1] = -t
        h[j+1, j] = -t
        Delta_mat[j, j+1] = delta
        Delta_mat[j+1, j] = -delta

    H_bdg = np.block([
        [h,              Delta_mat          ],
        [-Delta_mat.conj(), -h.T            ]
    ])
    return H_bdg

# ── Compare spectra: topological vs trivial ──
N = 20
H_topo = kitaev_bdg(N, mu=0.5, t=1.0, delta=1.0)
E_topo = np.sort(np.linalg.eigvalsh(H_topo))

H_triv = kitaev_bdg(N, mu=3.0, t=1.0, delta=1.0)
E_triv = np.sort(np.linalg.eigvalsh(H_triv))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(range(len(E_topo)), E_topo, 'bo', markersize=4)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax1.set_title(r'Topological Phase ($\mu$=0.5, t=$\Delta$=1)', fontsize=12)
ax1.set_xlabel('Eigenvalue index')
ax1.set_ylabel('Energy')
ax1.grid(axis='y', alpha=0.25)

ax2.plot(range(len(E_triv)), E_triv, 'ro', markersize=4)
ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax2.set_title(r'Trivial Phase ($\mu$=3.0, t=$\Delta$=1)', fontsize=12)
ax2.set_xlabel('Eigenvalue index')
ax2.set_ylabel('Energy')
ax2.grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig('kitaev_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

min_E_topo = np.min(np.abs(E_topo))
min_E_triv = np.min(np.abs(E_triv))
print(f"Min |E| in topological phase: {min_E_topo:.2e}")
print(f"Min |E| in trivial phase:     {min_E_triv:.2e}")
assert min_E_topo < 1e-10, "Topological phase must have zero-energy modes"
assert min_E_triv > 0.1, "Trivial phase must be gapped"
print("✅ Kitaev chain: zero modes in topological phase, gap in trivial phase")

---
## §4 — Topological Phase Diagram

The Kitaev chain has a **topological phase transition** at $|\mu| = 2t$
(for $\Delta \neq 0$):

| Parameter regime | Phase | Gap | Edge modes |
|:---:|:---:|:---:|:---:|
| $\lvert\mu\rvert < 2t$ | **Topological** | Open | Yes — Majorana zero modes |
| $\lvert\mu\rvert > 2t$ | **Trivial** | Open | No |
| $\lvert\mu\rvert = 2t$ | **Critical** | Closed | — |

At the critical point, the bulk energy gap closes and reopens, signalling a quantum phase
transition. The topological phase is characterized by a non-trivial **$\mathbb{Z}_2$ invariant**
(or equivalently, a winding number $\nu = 1$).

In [ ]:
N_sites = 50
t_val, delta_val = 1.0, 1.0
mu_values = np.linspace(-4, 4, 300)
gaps = []

for mu in mu_values:
    H = kitaev_bdg(N_sites, mu=mu, t=t_val, delta=delta_val)
    E = np.linalg.eigvalsh(H)
    gaps.append(np.min(np.abs(E)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mu_values / t_val, gaps, 'b-', linewidth=2)
ax.axvline(x=-2, color='r', linestyle='--', alpha=0.7,
           label=r'Phase boundary $|\mu|=2t$')
ax.axvline(x=2, color='r', linestyle='--', alpha=0.7)
ax.fill_between(mu_values / t_val, 0, max(gaps),
                where=(np.abs(mu_values) < 2 * t_val),
                alpha=0.1, color='green', label='Topological phase')
ax.set_xlabel(r'$\mu / t$', fontsize=12)
ax.set_ylabel(r'Energy gap  $\min|E|$', fontsize=12)
ax.set_title('Kitaev Chain — Topological Phase Diagram', fontsize=14)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('kitaev_phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

gap_at_boundary = np.interp(2.0, mu_values / t_val, gaps)
gap_in_topo = np.interp(0.0, mu_values / t_val, gaps)
print(f"Gap at mu/t = 0 (topological): {gap_in_topo:.4e}")
print(f"Gap at mu/t = 2 (boundary):    {gap_at_boundary:.4e}")
assert gap_at_boundary < 0.15, "Gap should nearly close at phase boundary"
print("✅ Phase diagram: gap closes at |mu| = 2t, confirming topological transition")

---
## §5 — Majorana Zero Modes

In the topological phase, the Kitaev chain hosts two **Majorana zero modes** (MZMs)
localized at opposite ends of the chain. These modes:

1. Have **exactly zero energy** (protected by particle-hole symmetry)
2. Are **exponentially localized** at the edges with decay length
   $\xi \sim 1/\bigl|\ln|\mu/2t|\bigr|$
3. Are **robust** against local perturbations that don't close the bulk gap

The spatial profile of a zero mode $\psi_0$ follows:

$$
|\psi_0(x)|^2 \sim e^{-2x/\xi}
$$

where $x$ is the distance from the edge. As the chain length $L \to \infty$, the two
edge modes become exactly degenerate at zero energy. For finite chains, the overlap
produces an exponentially small splitting $\delta E \sim e^{-L/\xi}$.

In [ ]:
N = 40
H = kitaev_bdg(N, mu=0.3, t=1.0, delta=1.0)
eigenvalues, eigenvectors = np.linalg.eigh(H)

# Find the two modes closest to zero energy
idx_sorted = np.argsort(np.abs(eigenvalues))
zero_idx = idx_sorted[:2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for k, (idx, ax) in enumerate(zip(zero_idx, axes)):
    psi = eigenvectors[:, idx]
    u, v = psi[:N], psi[N:]
    weight = np.abs(u)**2 + np.abs(v)**2

    ax.bar(range(N), weight, color='steelblue', edgecolor='black',
           alpha=0.8, linewidth=0.5)
    ax.set_xlabel('Site index', fontsize=11)
    ax.set_ylabel(r'$|\psi|^2$', fontsize=11)
    ax.set_title(f'Zero mode {k+1}  (E = {eigenvalues[idx]:.2e})', fontsize=12)
    ax.grid(axis='y', alpha=0.25)

plt.suptitle('Majorana Zero Modes — Edge Localization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('majorana_zero_modes.png', dpi=150, bbox_inches='tight')
plt.show()

for k, idx in enumerate(zero_idx):
    psi = eigenvectors[:, idx]
    u, v = psi[:N], psi[N:]
    weight = np.abs(u)**2 + np.abs(v)**2
    edge_weight = np.sum(weight[:5]) + np.sum(weight[-5:])
    total_weight = np.sum(weight)
    ratio = edge_weight / total_weight
    print(f"Zero mode {k+1}: {ratio:.1%} weight on edge sites (first + last 5)")
    assert ratio > 0.5, f"Zero mode {k+1} should be edge-localized"

print("✅ Majorana zero modes are exponentially localized at chain edges")

---
## §6 — Encoding Qubits in Majorana Modes

A pair of Majorana zero modes defines a **non-local fermion** whose occupation number
encodes one bit of quantum information. For $2N$ Majorana modes, we can encode $N$
qubits, subject to a global parity constraint.

### The 4-Majorana Qubit

Consider 4 Majorana modes $\gamma_0, \gamma_1, \gamma_2, \gamma_3$. Define parity
operators:

$$
P_{01} = i\gamma_0\gamma_1, \qquad P_{23} = i\gamma_2\gamma_3
$$

These satisfy $P_{ij}^2 = I$ and $[P_{01}, P_{23}] = 0$, so they can be simultaneously
diagonalized with eigenvalues $\pm 1$.

The **total fermion parity** $P = P_{01} P_{23}$ is conserved. Fixing $P = +1$ (even
parity) leaves a **2-dimensional qubit subspace** where:

$$
P_{01} \to Z_L \quad \text{(logical Z)}, \qquad i\gamma_1\gamma_2 \to X_L \quad
\text{(logical X)}
$$

The quantum information is stored in the **joint parity** of spatially separated Majorana
modes — no local measurement can access it, providing **intrinsic decoherence protection**.

In [ ]:
n_sites = 2
gammas = build_majorana_ops(n_sites)
dim = 2**n_sites

# ── Parity operators ──
P01 = 1j * gammas[0] @ gammas[1]
P23 = 1j * gammas[2] @ gammas[3]
P_total = P01 @ P23

assert np.allclose(P01 @ P01, np.eye(dim)), "P01^2 != I"
assert np.allclose(P23 @ P23, np.eye(dim)), "P23^2 != I"
assert np.allclose(P_total @ P_total, np.eye(dim)), "P^2 != I"
assert np.allclose(P01 @ P23, P23 @ P01), "[P01, P23] != 0"

# ── Extract even-parity qubit subspace ──
evals, evecs = np.linalg.eigh(P_total)
even_mask = np.isclose(evals, 1.0)
even_states = evecs[:, even_mask]

print(f"Even-parity subspace dimension: {even_states.shape[1]}")
assert even_states.shape[1] == 2, "Expected 2D qubit subspace"

# ── Logical operators in the qubit subspace ──
Z_L = even_states.conj().T @ P01 @ even_states
X_candidate = 1j * gammas[1] @ gammas[2]
X_L = even_states.conj().T @ X_candidate @ even_states

print(f"Logical Z (iγ₀γ₁):\n{np.round(Z_L.real, 3)}")
print(f"Logical X (iγ₁γ₂):\n{np.round(X_L.real, 3)}")

# Verify Pauli algebra
assert np.allclose(X_L @ Z_L + Z_L @ X_L, np.zeros((2, 2))), "{X_L, Z_L} != 0"
assert np.allclose(X_L @ X_L, np.eye(2)), "X_L^2 != I"
assert np.allclose(Z_L @ Z_L, np.eye(2)), "Z_L^2 != I"

print("✅ 4 Majorana modes → 1 logical qubit in even-parity sector")
print("   Logical Pauli algebra: {X_L, Z_L} = 0, X_L² = Z_L² = I")

---
## §7 — Non-Abelian Braiding

The defining feature of Majorana zero modes is their **non-Abelian exchange statistics**.
When two Majorana modes $\gamma_i$ and $\gamma_j$ are adiabatically exchanged (braided),
the system undergoes a unitary transformation:

$$
B_{ij} = \frac{I + \gamma_j\gamma_i}{\sqrt{2}}
= \exp\!\left(\frac{\pi}{4}\,\gamma_j\gamma_i\right)
$$

This braiding transforms the Majorana operators as:

$$
B_{ij}\, \gamma_i\, B_{ij}^\dagger = \gamma_j, \qquad
B_{ij}\, \gamma_j\, B_{ij}^\dagger = -\gamma_i
$$

while leaving all other $\gamma_k$ ($k \neq i, j$) unchanged.

### Non-Abelian Property

For three or more Majorana modes, braiding operations **do not commute**:

$$
B_{01}\, B_{12} \neq B_{12}\, B_{01}
$$

This non-commutativity is the hallmark of **non-Abelian anyons** and is what enables
braiding-based quantum computation. The braiding outcome depends on the **order** of
exchanges, not just which particles were exchanged.

In [ ]:
n_sites = 3  # 6 Majorana modes
gammas_3 = build_majorana_ops(n_sites)
dim_3 = 2**n_sites

def braid(gamma_list, i, j):
    """Braiding unitary B_{ij} = (I + gamma_j gamma_i) / sqrt(2)."""
    d = gamma_list[0].shape[0]
    return (np.eye(d) + gamma_list[j] @ gamma_list[i]) / np.sqrt(2)

B_01 = braid(gammas_3, 0, 1)
B_12 = braid(gammas_3, 1, 2)

# Verify unitarity
assert np.allclose(B_01 @ B_01.conj().T, np.eye(dim_3)), "B_01 not unitary"
assert np.allclose(B_12 @ B_12.conj().T, np.eye(dim_3)), "B_12 not unitary"

# ── Non-Abelian test: B_01 B_12 != B_12 B_01 ──
forward  = B_01 @ B_12
backward = B_12 @ B_01
commutator_norm = np.linalg.norm(forward - backward)
print(f"||B_01 B_12 - B_12 B_01|| = {commutator_norm:.6f}")
assert commutator_norm > 0.1, "Braiding must be non-Abelian"

# ── Verify braiding transformation rules ──
g0_new = B_01 @ gammas_3[0] @ B_01.conj().T
g1_new = B_01 @ gammas_3[1] @ B_01.conj().T
g2_new = B_01 @ gammas_3[2] @ B_01.conj().T

assert np.allclose(g0_new, gammas_3[1]), "B_01: gamma_0 should map to gamma_1"
assert np.allclose(g1_new, -gammas_3[0]), "B_01: gamma_1 should map to -gamma_0"
assert np.allclose(g2_new, gammas_3[2]), "B_01 should not affect gamma_2"

print("✅ Braiding is non-Abelian: B_01 B_12 ≠ B_12 B_01")
print("✅ Transformation rules: γ₀ → γ₁, γ₁ → −γ₀, γ₂ → γ₂")

---
## §8 — Topological Protection

The chief advantage of Majorana qubits is **topological protection**: the quantum
information encoded in spatially separated zero modes is immune to **local perturbations**.

Any local operator $V$ acting on a single site cannot distinguish between the degenerate
ground states because the qubit operators $P_{01} = i\gamma_0\gamma_1$ are **non-local** —
they involve Majorana modes at opposite ends of the chain.

Mathematically, for any local perturbation $V$ supported away from both edges:

$$
\langle \psi_0 | V | \psi_1 \rangle \sim e^{-L/\xi}
$$

where $L$ is the chain length and $\xi$ is the coherence length. The error rate is
**exponentially suppressed** in the system size — a qualitative improvement over
conventional error correction.

### Disorder Robustness

The topological phase survives moderate on-site disorder. The zero modes persist as long
as the disorder does not close the bulk gap (i.e., the system remains in the topological
phase).

In [ ]:
N = 40
rng = np.random.default_rng(42)

# ── Sweep disorder strength ──
disorder_strengths = [0.0, 0.3, 0.6, 0.9, 1.2]
all_spectra = {}

for W in disorder_strengths:
    disorder = W * rng.uniform(-1, 1, N)
    H = kitaev_bdg(N, mu=0.5, t=1.0, delta=1.0)
    for j in range(N):
        H[j, j] += disorder[j]
        H[N + j, N + j] -= disorder[j]    # BdG particle-hole symmetry
    E = np.sort(np.linalg.eigvalsh(H))
    all_spectra[W] = E

fig, ax = plt.subplots(figsize=(10, 5))
markers = ['o', 's', '^', 'D', 'v']
for (W, E), m in zip(all_spectra.items(), markers):
    ax.plot(range(len(E)), E, m, markersize=3, label=f'W = {W:.1f}', alpha=0.7)

ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Eigenvalue index', fontsize=11)
ax.set_ylabel('Energy', fontsize=11)
ax.set_title('Topological Protection — Spectrum Under On-Site Disorder', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig('topological_protection.png', dpi=150, bbox_inches='tight')
plt.show()

# Verify: zero modes survive moderate disorder
for W in [0.0, 0.3, 0.6]:
    min_E = np.min(np.abs(all_spectra[W]))
    print(f"W = {W:.1f}: min|E| = {min_E:.2e}")
    assert min_E < 0.05, f"Zero modes should survive at W={W}"

print("✅ Majorana zero modes are robust against local disorder (topological protection)")

---
## §9 — From Braiding to Quantum Gates

In the even-parity qubit subspace, braiding operations map to standard single-qubit
gates. For 4 Majorana modes encoding 1 qubit:

| Braiding | Majorana pair | Logical gate |
|:---:|:---:|:---:|
| $B_{01}$ | $\gamma_0, \gamma_1$ | $e^{i\pi Z_L / 4}$ (Phase gate) |
| $B_{12}$ | $\gamma_1, \gamma_2$ | $e^{i\pi X_L / 4}$ ($\sqrt{X}$-type) |
| $B_{23}$ | $\gamma_2, \gamma_3$ | $e^{-i\pi Z_L / 4}$ (Phase gate, conjugate) |

Braiding of Majorana modes generates the **Clifford group** — a crucial subset of quantum
gates that includes phase, rotation, and CNOT-like operations.

### Limitation: The Clifford Ceiling

Braiding alone cannot produce the **T gate** ($\pi/8$ rotation), which is required for
**universal quantum computation** (Solovay-Kitaev theorem). Topological quantum computers
need an additional non-topological ingredient — typically **magic state distillation** — to
achieve universality.

In [ ]:
# ── Project braiding gates into the qubit subspace ──
n_sites = 2
gammas_2 = build_majorana_ops(n_sites)
dim_2 = 2**n_sites

P_total = (1j * gammas_2[0] @ gammas_2[1]) @ (1j * gammas_2[2] @ gammas_2[3])
evals, evecs = np.linalg.eigh(P_total)
even_states = evecs[:, np.isclose(evals, 1.0)]

braids = {}
for i, j, label in [(0, 1, "B_01"), (1, 2, "B_12"), (2, 3, "B_23")]:
    B_full = braid(gammas_2, i, j)
    B_logical = even_states.conj().T @ B_full @ even_states
    braids[label] = B_logical
    print(f"{label} logical gate:")
    print(np.round(B_logical, 4))
    print()

# ── Verify gate properties ──
B01_L = braids["B_01"]
off_diag_01 = np.abs(B01_L[0, 1]) + np.abs(B01_L[1, 0])
print(f"B_01 off-diagonal magnitude: {off_diag_01:.6f} (diagonal = phase gate)")
assert off_diag_01 < 1e-10, "B_01 should be diagonal in Z basis"

B12_L = braids["B_12"]
off_diag_12 = np.abs(B12_L[0, 1]) + np.abs(B12_L[1, 0])
print(f"B_12 off-diagonal magnitude: {off_diag_12:.6f} (off-diagonal = rotation)")
assert off_diag_12 > 0.1, "B_12 should mix computational basis states"

for label, B in braids.items():
    assert np.allclose(B @ B.conj().T, np.eye(2)), f"{label} not unitary"

print("✅ Braiding produces Clifford gates in the logical qubit subspace")
print("   B_01 → phase-like (diagonal), B_12 → rotation-like (off-diagonal)")

---
## §10 — Topological Invariant: Winding Number

The topology of the Kitaev chain is captured by a **winding number** $\nu$ computed from
the momentum-space Hamiltonian. In the Fourier-transformed BdG form:

$$
H(k) = d_z(k)\,\tau_z + d_y(k)\,\tau_y
$$

where $\tau_{y,z}$ are Pauli matrices in particle-hole space and:

$$
d_z(k) = -\mu - 2t\cos k, \qquad d_y(k) = 2\Delta \sin k
$$

The **winding number** counts how many times the vector
$\mathbf{d}(k) = (d_y, d_z)$ winds around the origin as $k$ traverses the Brillouin zone:

$$
\nu = \frac{1}{2\pi} \oint_{\text{BZ}} d\theta(k), \qquad
\theta(k) = \arctan\!\frac{d_y(k)}{d_z(k)}
$$

| $\nu$ | Phase | Meaning |
|:---:|:---:|:---:|
| 1 | Topological | $\mathbf{d}(k)$ encircles the origin — edge modes exist |
| 0 | Trivial | $\mathbf{d}(k)$ does not encircle origin — no edge modes |

This is the **bulk-boundary correspondence**: a topological invariant computed from the
*bulk* (periodic) Hamiltonian predicts the existence of *boundary* (edge) modes.

In [ ]:
k_vals = np.linspace(0, 2 * np.pi, 1000, endpoint=False)
t_val, delta_val = 1.0, 1.0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cases = [
    (0.5, r'Topological ($\mu$=0.5)'),
    (2.0, r'Critical ($\mu$=2.0)'),
    (3.0, r'Trivial ($\mu$=3.0)')]

winding_numbers = []
for ax, (mu_val, title) in zip(axes, cases):
    dy = 2 * delta_val * np.sin(k_vals)
    dz = -mu_val - 2 * t_val * np.cos(k_vals)

    ax.plot(dz, dy, 'b-', linewidth=2)
    ax.plot(0, 0, 'r+', markersize=15, markeredgewidth=2)
    ax.set_xlabel(r'$d_z(k)$', fontsize=11)
    ax.set_ylabel(r'$d_y(k)$', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_aspect('equal')
    ax.grid(alpha=0.25)

    theta = np.arctan2(dy, dz)
    dtheta = np.diff(theta)
    dtheta = (dtheta + np.pi) % (2 * np.pi) - np.pi
    winding = np.sum(dtheta) / (2 * np.pi)
    winding_numbers.append(winding)

    ax.text(0.05, 0.95, f'$\\nu$ = {winding:.0f}', transform=ax.transAxes,
            fontsize=14, fontweight='bold', verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Topological Winding Number — d-vector Trajectories in the BZ',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('winding_number.png', dpi=150, bbox_inches='tight')
plt.show()

assert abs(abs(winding_numbers[0]) - 1.0) < 0.1, "Topological phase should have |nu|=1"
assert abs(winding_numbers[2]) < 0.1, "Trivial phase should have nu=0"

print(f"Winding numbers: nu = {winding_numbers[0]:.0f} (topo), "
      f"{winding_numbers[1]:.1f} (critical), {winding_numbers[2]:.0f} (trivial)")
print("✅ Bulk-boundary correspondence: nu=1 <-> edge modes, nu=0 <-> no edge modes")

---
## §11 — Microsoft's Topological Quantum Computing Roadmap

Microsoft's quantum computing strategy is built on **topological qubits** using Majorana
zero modes in semiconductor-superconductor heterostructures:

**Material platform:** InAs/Al nanowires under magnetic field — a 1D system engineered to
realize the Kitaev chain in a real material.

**Key milestones:**

1. **Topological gap protocol** (2022–2023): Experimental evidence for a topological phase
   in InAs/Al devices, verified via a systematic gap-closing/reopening signature.
2. **Majorana-based qubit** (2024–2025): Demonstration of a qubit encoded in Majorana zero
   modes with braiding-based gate operations.
3. **Scalable architecture**: Planned T-junction geometries enabling non-Abelian braiding
   in 2D networks of 1D wires.

**Why topology matters for error correction:**

| Property | Conventional qubit | Topological qubit |
|:---|:---|:---|
| Error source | Local noise (T₁, T₂) | Only non-local noise |
| Protection | Active error correction | Passive (built into physics) |
| Gate fidelity | Limited by decoherence | Limited by braiding precision |
| Overhead | Many physical qubits per logical | Fewer physical qubits needed |

The promise of topological quantum computing is that **the hardware itself provides a
level of error protection** that conventional qubits achieve only through massive software
overhead.

---
## 📋 Takeaways

| # | Concept | Key result |
|:---:|:---|:---|
| 1 | **Majorana operators** | Self-adjoint ($\gamma^\dagger = \gamma$), obey Clifford algebra $\{\gamma_i, \gamma_j\} = 2\delta_{ij}$ |
| 2 | **Dirac–Majorana map** | Each complex fermion $c$ decomposes into two real Majoranas: $c = (\gamma_0 + i\gamma_1)/2$ |
| 3 | **Kitaev chain** | 1D p-wave superconductor with topological ($\lvert\mu\rvert<2t$) and trivial ($\lvert\mu\rvert>2t$) phases |
| 4 | **Zero modes** | Exponentially localized at chain edges; energy splitting $\sim e^{-L/\xi}$ |
| 5 | **Qubit encoding** | 4 Majorana modes → 1 logical qubit in the even-parity sector |
| 6 | **Non-Abelian braiding** | $B_{01} B_{12} \neq B_{12} B_{01}$ — exchange order matters |
| 7 | **Topological protection** | Local perturbations cannot access non-local qubit information |
| 8 | **Clifford gates** | Braiding generates Clifford group; universality requires magic state distillation |
| 9 | **Winding number** | Bulk topological invariant $\nu \in \{0, 1\}$ predicts edge modes (bulk-boundary correspondence) |
| 10 | **Microsoft roadmap** | InAs/Al nanowires realize Kitaev chain; topological qubits aim to reduce error correction overhead |